In [1]:
#create a csv file for backend
import csv
import pandas as pd # To manage books dataframe 
from datetime import datetime, timedelta

In [2]:
# book dataframe contain all books in the library
# user dataframe  contains all users logs
# reservation dataframe  contain users who want to borrow a book not available now 
# borrowed books dataframe contain books that are borrowed by some user



books_df = pd.DataFrame(columns=[ "Title", "Author", "CopiesAvailable"])
users_df = pd.DataFrame(columns=["First Name", "Last Name","Age" , "Role", "Borrowed Books", "Fines"])
reservations_df = pd.DataFrame(columns=[ "Title", "Author", "Reserved By", "Reservation Date", "Queue Position", "Notification Sent"])
borrowed_books_df = pd.DataFrame(columns=["User", "Title", "Author", "Borrow Date", "Due Date"])
transaction_log_df = pd.DataFrame(columns=["Transaction Type", "User", "Title", "Author", "Transaction Date"])
havetoreturn_df = pd.DataFrame(columns=["User", "Book Title", "Author", "Original Due Date", "Days Overdue", "Fines"])


In [3]:

# Function to add books because the lib is empty
def add_book(title, author, copies):
    global books_df
    # Check if the book already exists (based on title and author)
    # if the book exist add it to a dataframe name existing_book .it will have one record if the book exist
    existing_book = books_df[(books_df['Title'] == title) & (books_df['Author'] == author)]

    # if existing_book not empty that means that the book exist then we will update the available copies instead of adding new record
    if not existing_book.empty:
        # If the book already exists, update the CopiesAvailable
        existing_index = existing_book.index[0]
        books_df.loc[existing_index, 'CopiesAvailable'] += copies # increment the number of copies of the book in the book_df
    else:
        # if the book not exist, add a new record with the generated BookID
        new_book_id = len(books_df) + 1
        books_df.loc[new_book_id] = {"Title": title, "Author": author, "CopiesAvailable": copies}

# Adding 10 programming books . A list of books to pass into our function
programming_books = [
    ("Python Programming", "John Doe", 5),
    ("Data Science with Python", "Jane Smith", 3),
    ("Machine Learning Basics", "Michael Brown", 4),
    ("Deep Learning with TensorFlow", "Anna White", 2),
    ("Artificial Intelligence: A Modern Approach", "Peter Green", 6),
    ("Introduction to Algorithms", "Thomas H. Cormen", 7),
    ("Clean Code", "Robert C. Martin", 5),
    ("Design Patterns", "Erich Gamma", 4),
    ("JavaScript: The Good Parts", "Douglas Crockford", 3),
    ("The Pragmatic Programmer", "Andrew Hunt", 8)
]


# loop into the list to call the function for every book to be added
for book in programming_books:
    add_book(*book)

# Display the Books DataFrame
print("Books DataFrame")
books_df



Books DataFrame


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,5
2,Data Science with Python,Jane Smith,3
3,Machine Learning Basics,Michael Brown,4
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,The Pragmatic Programmer,Andrew Hunt,8


In [4]:
# Function to add users
def add_user(first_name, last_name, age, gender, role="Regular", borrowed_books=None, fines=0):
    global users_df
    borrowed_books = borrowed_books if borrowed_books is not None else pd.DataFrame(columns=["Book Title", "First Date to Borrow", "Last Date to Return"])
    
    # Check if the user already exists based on First Name, Last Name, and Age
    existing_user = users_df[(users_df['First Name'] == first_name) & 
                             (users_df['Last Name'] == last_name) & 
                             (users_df['Age'] == age)]
    
    if not existing_user.empty:
        print(f"User {first_name} {last_name} already exists.")
        return

    # Add the new user to the DataFrame
    new_user = pd.DataFrame({
        "First Name": [first_name],
        "Last Name": [last_name],
        "Age": [age],
        "Gender": [gender],
        "Role": [role],
        "Borrowed Books":  str(borrowed_books), 
        "Fines": [fines]
    })
    users_df = pd.concat([users_df, new_user], ignore_index=True)

# Adding 10 users
users_list = [
    ("Leen", "Samman", 20, "Female"),
    ("Amal", "Taha", 34, "Female"),
    ("Amany", "Awwad", 22, "Female"),
    ("Ahmad", "Bilal", 55, "Male"),
    ("Karam", "Jallad", 19, "Male"),
    ("Qamar", "Sayeed", 33, "Female"),
    ("Hayat", "Najjar", 19, "Female"),
    ("Tamara", "Mousa", 36, "Female"),
    ("Omar", "Bilal", 28, "Male"),
    ("Amjad", "Rami", 41, "Male")
]

# Add users to the DataFrame
for user in users_list:
    add_user(user[0], user[1], user[2], user[3])

# Save the users_df DataFrame to a CSV file
users_df.to_csv('users.csv', index=False)

print("Users DataFrame has been saved to 'users.csv'.")

users_df

Users DataFrame has been saved to 'users.csv'.


,First Name,Last Name,Age,Role,Borrowed Books,Fines,Gender
0,Leen,Samman,20,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
1,Amal,Taha,34,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
2,Amany,Awwad,22,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
3,Ahmad,Bilal,55,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
4,Karam,Jallad,19,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
5,Qamar,Sayeed,33,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
6,Hayat,Najjar,19,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
7,Tamara,Mousa,36,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
8,Omar,Bilal,28,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
9,Amjad,Rami,41,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male


In [5]:

# A class to create entities involved in the library managment system (regular users + Admins)
class users:
    def __init__(self,first_name,last_name,age,gender,role="Regular"): #first name,last name ,age ,gender and role are needed to create the user
        self.first_name = first_name
        self.last_name = last_name
        self.age = age
        self.gender = gender
        self.role = role
        self.borrowed_books = pd.DataFrame(columns=["Book Title", "Author","Borrow Date", "Return Date"])  # when create the user he/she have no borrowed books yet
        self.havetoreturn = pd.DataFrame(columns=["Book Title", "Original Due Date", "Days Overdue"]) #This DataFrame will store books that the user is late to return, including the necessary details like the book title, the original due date, and the number of days overdue.
        self.reserved_books = pd.DataFrame(columns=["Book Title", "Reservation Date"])  # if the book not available the user can reserve it .......
        self.fines = 0  # Start with zero fines because when creating a user . user did not borrow any book yet
       
        # As soon a user is created will be add  to the users_df
        # Add user to users_df
        new_user = pd.DataFrame({
            "First Name": [self.first_name],
            "Last Name": [self.last_name],
            "Age": [self.age],
            "Gender": [self.gender],
            "Role": [self.role],
            "Borrowed Books": [self.borrowed_books],
            "Fines": [self.fines]
        })
        global users_df
        users_df = pd.concat([users_df, new_user], ignore_index=True)
        
       

        


#----------------------------------------------------------------------------------------------------------------------------------------------------



    # Method to check and update overdue books (automatically called)
    # function automatically tracks books that have passed their return dates and marks them as overdue.
    # _update_global_overdue_books is automatically called during functions like borrow_book, return_book, user_login, daily checks, or system startup to update overdue books and apply fines.
    
    def _update_global_overdue_books(self):
        global havetoreturn_df
        current_date = datetime.now()
        print(f"Current Date: {current_date}")
    
        # Ensure dates are in datetime format
        self.borrowed_books["Return Date"] = pd.to_datetime(self.borrowed_books["Return Date"])
    
        # overdue_books are the books that are overdue from the user borrowed books
        overdue_books = self.borrowed_books[self.borrowed_books["Return Date"] < current_date]  # the user specify the borrowed_books["Return Date"] when borrowing a book
        
        print(f"Identified Overdue Books:\n{overdue_books}")
    
        # A loop for fine calculation
        # access the "Return Date" for the current book in each row. The "Return Date" was set when the book was borrowed.
        # then we calculate the time difference between the current date (current_date) and the "Return Date" for the book.
        # .days is an attribute of a timedelta object returns the difference in days between the two dates to calculate overdue days.
        for _, row in overdue_books.iterrows():  # no index "_" to ignore it
            days_overdue = (current_date - row["Return Date"]).days
    
            # Check if the book is already in the global overdue DataFrame to avoid duplication
            # if not exist
            if not ((havetoreturn_df["User"] == f"{self.first_name} {self.last_name}") & 
                    (havetoreturn_df["Book Title"] == row["Book Title"]) & 
                    (havetoreturn_df["Author"] == row["Author"])).any():
                # then add the book to overdue_entry dataframe then it will be added to the global havetoreturn_df  
                overdue_entry = pd.DataFrame([{
                    "User": f"{self.first_name} {self.last_name}",
                    "Book Title": row["Book Title"],
                    "Author": row["Author"],
                    "Original Due Date": row["Return Date"].strftime("%Y-%m-%d"),
                    "Days Overdue": days_overdue,
                    "Fines": days_overdue  # Fine calculated based on days overdue
                }])
    
                # The global havetoreturn_df DataFrame is updated to include all overdue books across all users
                # we already calculate the fines for overdue books based on the number of days past the return date. 
                # This fine is added to the user's total fines.
                havetoreturn_df = pd.concat([havetoreturn_df, overdue_entry], ignore_index=True)
                self.fines += days_overdue
                print(f"Added '{row['Book Title']}' to overdue with {days_overdue} days overdue.")
            else:
                print(f"{row['Book Title']} is already recorded as overdue.")




#----------------------------------------------------------------------------------------------------------------------------------------------------
    


    # user want to borrow a book .he/she should pass the book name and auther and how many days he/she want to borrow the book
    # user cannot borrow the book for more than 30 days
    # ex. leen.borrow_book("Python Programming", "John Doe", 15) 
    def borrow_book(self, book_title, book_author, borrow_duration=30):
        borrow_date = datetime.now()
        # Ensure the borrow duration does not exceed 30 days. if it does then set to 30
        borrow_duration = min(borrow_duration, 30)
        # the book need to be available so the user can borrow it
        # first check if the book  available by the book title and its auther.
        # we will  filter the rows of books_df where the Title and Author match the 
        # book_title and book_author provided by the user .book_entry is a dataframe that contain matched books
        # only one book should be matched 
        book_entry = books_df[(books_df["Title"] == book_title) & (books_df["Author"] == book_author)]

        # if match then book_entry is not empty .the book is in the library and we need to check if their exist a copy from the book
        if not book_entry.empty:
            book_index = book_entry.index[0]     # the dataframe book entry should have one record but if their is duplicates for some reason maybe spelling. the first record will be retrieved
            if books_df.at[book_index, "CopiesAvailable"] > 0:                # if there is a copy available
                books_df.at[book_index, "CopiesAvailable"] -= 1               #then Decrement the available copies by one
                                                                            # at faster than iloc when accessin a single scaler value
                
                return_date = borrow_date + timedelta(days=borrow_duration)    # Calculate the return date

                
                # Adding the borrowed book details to the user's DataFrame
                self.borrowed_books.loc[len(self.borrowed_books)] = [
                    book_title,
                    book_author,
                    borrow_date.strftime("%Y-%m-%d"),
                    return_date.strftime("%Y-%m-%d")
                ]
                # the is book available . the user is going to borrow the book then -> 
                # 1) Add the borrowed book details to the borrowed_books_df
                borrowed_books_df.loc[len(borrowed_books_df)] = [
                    f"{self.first_name} {self.last_name}",
                    book_title,
                    book_author,
                    borrow_date.strftime("%Y-%m-%d"),
                    return_date.strftime("%Y-%m-%d")
                ]

                
                # 2) Log the borrowing transaction
                transaction_log_df.loc[len(transaction_log_df)] = [
                    "Borrow",
                    f"{self.first_name} {self.last_name}",
                    book_title,
                    book_author,
                    borrow_date.strftime("%Y-%m-%d")
                ]


                
                print(f"{book_title} by {book_author} has been borrowed until {return_date.strftime('%Y-%m-%d')}.")

                # Check if the user is late returning the book
                current_date = datetime.now()
                if current_date > return_date:
                    late_days = (current_date - return_date).days  # Calculate how many days the user is late
                    fine = late_days                            # One dollar fine per late day *1
                    print(f"You are {late_days} days late to return the book. A fine of ${fine} has been applied.")
                    
            
            else:
                print(f"No copies of '{book_title}' by {book_author} are currently available.") # there is a match but no available copies
        else:
            print(f"'{book_title}' by {book_author} is not in the library.")   # if there is no match. book entry is empty

        # Update global overdue books after borrowing
        self._update_global_overdue_books()


#----------------------------------------------------------------------------------------------------------------------------------------------------

    def return_book(self, book_title, book_author):
        # search for the book in the borrowed books df to update it
        # when find it assign the book to borrowed_entry
        borrowed_entry = borrowed_books_df[
            (borrowed_books_df["User"] == f"{self.first_name} {self.last_name}") & 
            (borrowed_books_df["Title"] == book_title) & 
            (borrowed_books_df["Author"] == book_author)
        ]
        
        # when we find the book then the borrowed_entry is not empty ->
        # 1) drop the record that tell that the user borrowed the book 
        if not borrowed_entry.empty:
            borrowed_books_df.drop(borrowed_entry.index, inplace=True)
            self.borrowed_books = self.borrowed_books[
                ~((self.borrowed_books["Book Title"] == book_title) & 
                  (self.borrowed_books["Author"] == book_author))
            ]
            
            # 2) update the books_df . the book is return then there is another available copy
            book_entry = books_df[(books_df["Title"] == book_title) & (books_df["Author"] == book_author)]
            if not book_entry.empty:
                book_index = book_entry.index[0]
                books_df.at[book_index, "CopiesAvailable"] += 1
                print(f"{book_title} by {book_author} has been returned.")
    
                # 3) Log the returning transaction
                transaction_log_df.loc[len(transaction_log_df)] = [
                    "Return",
                    f"{self.first_name} {self.last_name}",
                    book_title,
                    book_author,
                    datetime.now().strftime("%Y-%m-%d")
                ]
            else:
                print(f"Error: The book '{book_title}' by {book_author} was not found in the library records.")
            
            # 4) check if anyone reserved the book to notify the first on the queue if they are not notified
            reservations_for_book = reservations_df[
                (reservations_df["Title"] == book_title) & 
                (reservations_df["Author"] == book_author) & 
                (reservations_df["Notification Sent"] == False)
            ]
            if not reservations_for_book.empty:
                first_in_queue = reservations_for_book.loc[reservations_for_book["Queue Position"].idxmin()]
                print(f"Notification: '{book_title}' by {book_author} is now available for {first_in_queue['Reserved By']}.")
                reservations_df.at[first_in_queue.name, "Notification Sent"] = True
                
                # Simulate the borrower borrowing the book
                borrower_name = first_in_queue['Reserved By']
                borrower_first_name, borrower_last_name = borrower_name.split()
                
                # Find the user in the users_df
                borrower = users_df[
                    (users_df["First Name"] == borrower_first_name) & 
                    (users_df["Last Name"] == borrower_last_name)
                ]
                
                if not borrower.empty:
                    # Simulate the borrower borrowing the book
                    borrower_user = users(borrower_first_name, borrower_last_name, borrower['Age'].iloc[0], borrower['Gender'].iloc[0])
                    borrower_user.borrow_book(book_title, book_author)
                    
        else:
            print(f"No record of borrowing '{book_title}' by {book_author} found.")
        
        # Update global overdue books after returning
        self._update_global_overdue_books()
    


    
    
#----------------------------------------------------------------------------------------------------------------------------------------------------    
    
    
    #  function for user to display his/her overdue books
    def display_my_overdue_books(self):
        my_overdue_books = havetoreturn_df[havetoreturn_df["User"] == f"{self.first_name} {self.last_name}"]
        if not my_overdue_books.empty:
            print("Your Overdue Books:")
            print(my_overdue_books)
        else:
            print("You have no overdue books.")

    

 #----------------------------------------------------------------------------------------------------------------------------------------------------  



    #user should be able to see all the books he/she currently borrowing
    def view_borrowed_books(self):
        """Display all currently borrowed books by the user."""
        user_borrowed_books = borrowed_books_df[borrowed_books_df["User"] == f"{self.first_name} {self.last_name}"]
        if user_borrowed_books.empty:
            print("No books currently borrowed.")
        else:
            print("Currently Borrowed Books:")
            print(user_borrowed_books)
    
#----------------------------------------------------------------------------------------------------------------------------------------------------  


    # user may want to reserve a book if all copies are currently borrowed
    # the reservation date will be the current date 
    def reserve_book(self, book_title, book_author):
        # check if there is no copy available . we need to find the book in book_df
        book_entry = books_df[(books_df["Title"] == book_title) & (books_df["Author"] == book_author)]
    
        # if we find the book and the copies are 0 the user will wait on a queue
        # the reservation date will be the current date 
        if not book_entry.empty:
            book_index = book_entry.index[0]
            if books_df.at[book_index, "CopiesAvailable"] == 0:
                # Automatically set the reservation date to the current date
                reservation_date = datetime.now().strftime("%Y-%m-%d")
                
                # Add the reservation to the user reserved_books df
                new_reservation = pd.DataFrame([{
                    "Book Title": book_title,
                    "Reservation Date": reservation_date
                }])
                self.reserved_books = pd.concat([self.reserved_books, new_reservation], ignore_index=True)
                
                # Add the reservation to the global reservations DataFrame
                reservations_df.loc[len(reservations_df)] = [
                    book_title, 
                    book_author, 
                    f"{self.first_name} {self.last_name}", 
                    reservation_date, 
                    None,  # Queue position will be determined after sorting
                    False  # Notification Sent flag
                ]
                
                # call a function to Sort the reservations by Reservation Date and update the queue positions
                self.update_queue_positions(book_title, book_author)
                
                print(f"'{book_title}' by {book_author} has been reserved.")
            else:
                print(f"'{book_title}' by {book_author} is currently available, no need to reserve.")
        else:
            print(f"'{book_title}' by {book_author} does not exist in the library.")


    # a function to Sort the reservations by Reservation Date and update the queue positions
    def update_queue_positions(self, book_title, book_author):
        book_reservations = reservations_df[(reservations_df["Title"] == book_title) & 
                                            (reservations_df["Author"] == book_author)]
        
        # Sort by reservation date and reset queue positions
        sorted_reservations = book_reservations.sort_values(by="Reservation Date").reset_index(drop=True)
        
        # Update the queue positions in the reservations DataFrame
        for position, (index, reservation) in enumerate(sorted_reservations.iterrows(), start=1):
            reservations_df.at[index, "Queue Position"] = position
        
        print(f"Queue positions updated for '{book_title}' by {book_author}.")

#-----------------------------------------------------------------------------------------------------------------------------------


    # Method to display all reserved books for the user
    def display_reserved_books(self):
        """Display all books the user has reserved."""
        if self.reserved_books.empty:
            print("No books currently reserved.")
        else:
            print("Reserved Books:")
            print(self.reserved_books.to_string(index=False))

#----------------------------------------------------------------------------------------------------------------------------------------------------
# a user may want to cancel reservations . if so and he is notified that the book is available 
# the nontification will be sent to the next person on the queue 
    def cancel_reservation(self, book_title, book_author):
        # Find the user's reservation in the global reservations DataFrame
        reserved_book = reservations_df[(reservations_df["Title"] == book_title) & 
                                        (reservations_df["Author"] == book_author) & 
                                        (reservations_df["Reserved By"] == f"{self.first_name} {self.last_name}")]
        # if we find the reservation then we need to 1) Remove the user from the queue
        if not reserved_book.empty: 
            reservations_df.drop(reserved_book.index, inplace=True)
            
            # 2) Remove the reservation from the user's reserved_books DataFrame
            self.reserved_books = self.reserved_books[self.reserved_books["Book Title"] != book_title]
            
            # 3) Update the queue positions for other users
            queue_position = reserved_book.iloc[0]["Queue Position"]
            reservations_df.loc[(reservations_df["Title"] == book_title) & 
                                (reservations_df["Author"] == book_author) & 
                                (reservations_df["Queue Position"] > queue_position), 
                                "Queue Position"] -= 1
            
            print(f"Reservation for '{book_title}' by {book_author} has been canceled.")
            
            # 4) Check if the book is currently available
            book_entry = books_df[(books_df["Title"] == book_title) & (books_df["Author"] == book_author)]
            if not book_entry.empty and books_df.at[book_entry.index[0], "CopiesAvailable"] > 0:
                # If the book is available, notify the first user in the queue
                next_reservation = reservations_df[(reservations_df["Title"] == book_title) & 
                                                   (reservations_df["Author"] == book_author)].sort_values("Queue Position").head(1)
                if not next_reservation.empty:
                    next_user = next_reservation.iloc[0]["Reserved By"]
                    print(f"Notification: '{book_title}' by {book_author} is now available for {next_user}.")
                    reservations_df.at[next_reservation.index[0], "Notification Sent"] = True
            else:
                print(f"'{book_title}' by {book_author} is still borrowed by another user, but your cancellation was successful.")
        else:
            print(f"No reservation found for '{book_title}' by {book_author}.")


#----------------------------------------------------------------------------------------------------------------------------------------------------

# Function to display the user's details, including basic information, borrowed books, reserved books, and overdue books.

    def display_user_details(self):
        """Display the details of the user."""
        print(f"User Details for {self.first_name} {self.last_name}:")
        print(f"Age: {self.age}")
        print(f"Gender: {self.gender}")
        print(f"Role: {self.role}")
        print(f"Fines: ${self.fines}")
    
        print("\nBorrowed Books:")
        if self.borrowed_books.empty:
            print("No books currently borrowed.")
        else:
            print(self.borrowed_books.to_string(index=False))
    
        print("\nReserved Books:")
        if self.reserved_books.empty:
            print("No books currently reserved.")
        else:
            print(self.reserved_books.to_string(index=False))
    
        print("\nOverdue Books:")
        if self.havetoreturn.empty:
            print("No overdue books.")
        else:
            print(self.havetoreturn.to_string(index=False))
    




In [6]:

"""
All the critical aspects of the library management system are within the full visibility of the admin:
Books, Users, Reservations, Borrowed Books, and Transactions.This will ensure a comprehensive level of control 
and oversight, very essential for effective library managemen
"""

class Admin(users):
    def __init__(self, first_name, last_name, age, gender):
        # Call the __init__ method of the Users class
        super().__init__(first_name, last_name, age, gender, role="Admin")



    # An admin can add a book by passing the title of the book and the author and how many copies of the book he/she want to add

    # An admin can add a book by passing the title of the book and the author and how many copies of the book he/she want to add
    def add_book(self, title, author, copies):
        existing_book = books_df[(books_df["Title"] == title) & (books_df["Author"] == author)]
        
        if existing_book.empty:
            new_book = {"Title": title, "Author": author, "CopiesAvailable": copies}
            books_df.loc[len(books_df)] = new_book
            print(f"Book '{title}' by {author} has been added with {copies} copies.")
        else:
            book_index = existing_book.index[0]
            books_df.at[book_index, "CopiesAvailable"] += copies
            print(f"Copies of '{title}' by {author} have been increased by {copies}. Total copies available: {books_df.at[book_index, 'CopiesAvailable']}")

    
    # admin can delete the record for the book by passing the title and auther
    def delete_book_record(self, title, author):
        book_entry = books_df[(books_df["Title"] == title) & (books_df["Author"] == author)]
        
        if not book_entry.empty:
            books_df.drop(book_entry.index, inplace=True)
            print(f"All copies of '{title}' by {author} have been deleted from the library.")
        else:
            print(f"Book '{title}' by {author} does not exist in the library.") # book not exist to be deleted


    # One of the copies may be damaged so admin can delete one copy not all the book record
    def decrement_copies(self, title, author):
        book_entry = books_df[(books_df["Title"] == title) & (books_df["Author"] == author)]
        
        if not book_entry.empty:
            book_index = book_entry.index[0]
            available_copies = books_df.at[book_index, "CopiesAvailable"]
            
            if available_copies > 1:
                books_df.at[book_index, "CopiesAvailable"] -= 1
                print(f"One copy of '{title}' by {author} has been deleted. {available_copies - 1} copies remain.")
            else:
                # If there's only one copy left, notify admin to  delete the entire record
                print(f"Only one copy of '{title}' by {author} remains. Consider deleting the entire record instead.")
        else:
            print(f"Book '{title}' by {author} does not exist in the library.") # book  not exist to be deleted



    # admin can display all books 
    def display_all_books(self):
        if not books_df.empty:
            print("All Books in the Library:")
            print(books_df.to_string(index=False))
        else:
            print("No books available in the library.")


    #admin can display all users
    def display_all_users(self):
        """Display data for all users."""
        if not users_df.empty:
            print("All Users Data:")
            print(users_df.to_string(index=False))
        else:
            print("No users found.")




    # admin can display all current reservations
    def display_all_reservations(self):
        if not reservations_df.empty:
            print("All Reservations:")
            print(reservations_df.to_string(index=False))
        else:
            print("No reservations found.")




    # admin can display all borrowed books
    def display_all_borrowed_books(self):
        if not borrowed_books_df.empty:
            print("All Borrowed Books:")
            print(borrowed_books_df.to_string(index=False))
        else:
            print("No borrowed books found.")



    # admin can display all transactions done by users
    def display_all_transactions(self):
        if not transaction_log_df.empty:
            print("All Transactions:")
            print(transaction_log_df.to_string(index=False))
        else:
            print("No transactions found.")



# Test Cases

In [8]:
# Test Case 1: Add Books to the Library

# Add a new book
add_book("Learning Python", "Mark Lutz", 3)
print("After adding 'Learning Python' by 'Mark Lutz':")
books_df

After adding 'Learning Python' by 'Mark Lutz':


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,5
2,Data Science with Python,Jane Smith,3
3,Machine Learning Basics,Michael Brown,4
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,The Pragmatic Programmer,Andrew Hunt,8


In [9]:
# Add an existing book with additional copies
add_book("Python Programming", "John Doe", 2)
print("\nAfter adding 2 more copies of 'Python Programming' by 'John Doe':")
books_df


After adding 2 more copies of 'Python Programming' by 'John Doe':


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,7
2,Data Science with Python,Jane Smith,3
3,Machine Learning Basics,Michael Brown,4
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,The Pragmatic Programmer,Andrew Hunt,8


In [10]:
# Test Case 2: Add Users to the System

# Add a new user
add_user("John", "Smith", 25, "Male")
print("\nAfter adding 'John Smith':")
users_df


After adding 'John Smith':


,First Name,Last Name,Age,Role,Borrowed Books,Fines,Gender
0,Leen,Samman,20,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
1,Amal,Taha,34,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
2,Amany,Awwad,22,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
3,Ahmad,Bilal,55,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
4,Karam,Jallad,19,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
5,Qamar,Sayeed,33,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
6,Hayat,Najjar,19,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
7,Tamara,Mousa,36,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
8,Omar,Bilal,28,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
9,Amjad,Rami,41,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male


In [11]:
# Attempt to add the same user again
add_user("John", "Smith", 25, "Male")
print("\nAfter trying to add 'John Smith' again:")
users_df


User John Smith already exists.

After trying to add 'John Smith' again:


,First Name,Last Name,Age,Role,Borrowed Books,Fines,Gender
0,Leen,Samman,20,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
1,Amal,Taha,34,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
2,Amany,Awwad,22,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
3,Ahmad,Bilal,55,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
4,Karam,Jallad,19,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
5,Qamar,Sayeed,33,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
6,Hayat,Najjar,19,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
7,Tamara,Mousa,36,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
8,Omar,Bilal,28,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
9,Amjad,Rami,41,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male


In [12]:
# Test Case 3: Borrow a Book
# there is a case that the code does not handle
# the case indicate the the user could borrow the same at the same time
# for every cell run the copies will be decreased

# Create a user instance
john_smith = users("John", "Smith", 25, "Male")

# John borrows "Python Programming" by "John Doe" for 15 days
john_smith.borrow_book("Python Programming", "John Doe", 15)
print("\nAfter 'John Smith' borrows 'Python Programming':")
print(borrowed_books_df)
books_df


Python Programming by John Doe has been borrowed until 2024-09-06.
Current Date: 2024-08-22 16:47:30.747700
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []

After 'John Smith' borrows 'Python Programming':
         User               Title    Author Borrow Date    Due Date
0  John Smith  Python Programming  John Doe  2024-08-22  2024-09-06


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,6
2,Data Science with Python,Jane Smith,3
3,Machine Learning Basics,Michael Brown,4
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,The Pragmatic Programmer,Andrew Hunt,8


In [13]:
# Test Case 4: Return a Book

# John returns "Python Programming" by "John Doe"
john_smith.return_book("Python Programming", "John Doe")
print("\nAfter 'John Smith' returns 'Python Programming':")
borrowed_books_df



Python Programming by John Doe has been returned.
Current Date: 2024-08-22 16:47:30.766866
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []

After 'John Smith' returns 'Python Programming':


,User,Title,Author,Borrow Date,Due Date


In [14]:
books_df


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,7
2,Data Science with Python,Jane Smith,3
3,Machine Learning Basics,Michael Brown,4
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,The Pragmatic Programmer,Andrew Hunt,8


In [15]:
transaction_log_df

,Transaction Type,User,Title,Author,Transaction Date
0,Borrow,John Smith,Python Programming,John Doe,2024-08-22
1,Return,John Smith,Python Programming,John Doe,2024-08-22


In [16]:
# Test Case 5: Reserve a Book

# First, we'll borrow all copies of "Machine Learning Basics" by "Michael Brown" to ensure no copies are available.
leen_samman = users("Leen", "Samman", 20, "Female")
for _ in range(4):  # Assuming there are 4 copies available
    leen_samman.borrow_book("Machine Learning Basics", "Michael Brown", 10)

# Now, John tries to reserve "Machine Learning Basics"
john_smith.reserve_book("Machine Learning Basics", "Michael Brown")
print("\nAfter 'John Smith' reserves 'Machine Learning Basics':")
reservations_df


Machine Learning Basics by Michael Brown has been borrowed until 2024-09-01.
Current Date: 2024-08-22 16:47:30.799280
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
Machine Learning Basics by Michael Brown has been borrowed until 2024-09-01.
Current Date: 2024-08-22 16:47:30.804259
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
Machine Learning Basics by Michael Brown has been borrowed until 2024-09-01.
Current Date: 2024-08-22 16:47:30.808260
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
Machine Learning Basics by Michael Brown has been borrowed until 2024-09-01.
Current Date: 2024-08-22 16:47:30.811259
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
Queue positions updated for 'Machine Learning Basics' by Michael Brown.
'Machine Learning Basics

,Title,Author,Reserved By,Reservation Date,Queue Position,Notification Sent
0,Machine Learning Basics,Michael Brown,John Smith,2024-08-22,1,False


In [17]:
# Test Case 6: Cancel a Reservation

# Step 1: John reserves "Machine Learning Basics" by "Michael Brown"
john_smith.reserve_book("Machine Learning Basics", "Michael Brown")

# Step 2: John cancels his reservation for "Machine Learning Basics"
john_smith.cancel_reservation("Machine Learning Basics", "Michael Brown")
print("\nAfter 'John Smith' cancels his reservation for 'Machine Learning Basics':")
reservations_df


Queue positions updated for 'Machine Learning Basics' by Michael Brown.
'Machine Learning Basics' by Michael Brown has been reserved.
Reservation for 'Machine Learning Basics' by Michael Brown has been canceled.
'Machine Learning Basics' by Michael Brown is still borrowed by another user, but your cancellation was successful.

After 'John Smith' cancels his reservation for 'Machine Learning Basics':


,Title,Author,Reserved By,Reservation Date,Queue Position,Notification Sent


In [18]:
# Test Case 7: Multiple Reservations with Three Users and Assigning a Book to the Next User in the Queue

# Step 1: Ensure all copies of "Machine Learning Basics" are borrowed
leen_samman = users("Leen", "Samman", 20, "Female")
for _ in range(4):  # Assuming there are 4 copies available
    leen_samman.borrow_book("Machine Learning Basics", "Michael Brown", 10)

# Step 2: John Smith reserves "Machine Learning Basics"
john_smith.reserve_book("Machine Learning Basics", "Michael Brown")

# Step 3: User 2 reserves "Machine Learning Basics"
user2 = users("layla", "ali", 30, "Male")
user2.reserve_book("Machine Learning Basics", "Michael Brown")

# Step 4: User 3 reserves "Machine Learning Basics"
user3 = users("dina", "karam", 25, "Female")
user3.reserve_book("Machine Learning Basics", "Michael Brown")

# Check the reservation queue before the book is returned
print("\nReservation queue before returning the book:")
reservations_df



No copies of 'Machine Learning Basics' by Michael Brown are currently available.
Current Date: 2024-08-22 16:47:30.842950
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
No copies of 'Machine Learning Basics' by Michael Brown are currently available.
Current Date: 2024-08-22 16:47:30.844950
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
No copies of 'Machine Learning Basics' by Michael Brown are currently available.
Current Date: 2024-08-22 16:47:30.846949
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
No copies of 'Machine Learning Basics' by Michael Brown are currently available.
Current Date: 2024-08-22 16:47:30.847948
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
Queue positions updated for 'Machine Learning Basics' by Michael Brown.
'Machine

,Title,Author,Reserved By,Reservation Date,Queue Position,Notification Sent
0,Machine Learning Basics,Michael Brown,John Smith,2024-08-22,1,False
1,Machine Learning Basics,Michael Brown,layla ali,2024-08-22,2,False
2,Machine Learning Basics,Michael Brown,dina karam,2024-08-22,3,False


In [19]:
# Step 5: Leen Samman returns a copy of "Machine Learning Basics"
leen_samman.return_book("Machine Learning Basics", "Michael Brown")

# Check the reservation queue and borrowed books after the book is returned
print("\nReservation queue after returning the book:")
reservations_df


Machine Learning Basics by Michael Brown has been returned.
Notification: 'Machine Learning Basics' by Michael Brown is now available for John Smith.
Machine Learning Basics by Michael Brown has been borrowed until 2024-09-21.
Current Date: 2024-08-22 16:47:30.879198
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
Current Date: 2024-08-22 16:47:30.881198
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []

Reservation queue after returning the book:


,Title,Author,Reserved By,Reservation Date,Queue Position,Notification Sent
0,Machine Learning Basics,Michael Brown,John Smith,2024-08-22,1,True
1,Machine Learning Basics,Michael Brown,layla ali,2024-08-22,2,False
2,Machine Learning Basics,Michael Brown,dina karam,2024-08-22,3,False


In [20]:
print("\nBorrowed books after returning the book:")
borrowed_books_df


Borrowed books after returning the book:


,User,Title,Author,Borrow Date,Due Date
0,John Smith,Machine Learning Basics,Michael Brown,2024-08-22,2024-09-21


In [21]:
# The overdue book handling test case was not executed because the due date 
# falls on the current date, and the system correctly does not apply fines 
# when the due date is today.



In [22]:
# Test Case: Admin Deletes a Book Record

# Step 1: Admin adds "Data Science with Python" by "Jane Smith" to the library
admin = Admin("Admin", "User", 40, "Male")
admin.add_book("Data Science with Python", "Jane Smith", 5)

# Step 2: User (laya abed) borrows the book
laya_abed = users("Laya", "abed", 22, "Female")
laya_abed.borrow_book("Data Science with Python", "Jane Smith", 10)

# Step 3: Another user (Zaina Abunasser) reserves the book
sima_abunassar = users("Sima", "Abunassar", 23, "Female")
sima_abunassar.reserve_book("Data Science with Python", "Jane Smith")

# Step 4: Admin deletes the "Data Science with Python" book record
admin.delete_book_record("Data Science with Python", "Jane Smith")

# Step 5: Validate that the book record is deleted
print("\nBooks DataFrame after deletion:")
books_df





Copies of 'Data Science with Python' by Jane Smith have been increased by 5. Total copies available: 8
Data Science with Python by Jane Smith has been borrowed until 2024-09-01.
Current Date: 2024-08-22 16:47:30.923241
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
'Data Science with Python' by Jane Smith is currently available, no need to reserve.
All copies of 'Data Science with Python' by Jane Smith have been deleted from the library.

Books DataFrame after deletion:


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,7
3,Machine Learning Basics,Michael Brown,0
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,The Pragmatic Programmer,Andrew Hunt,8
11,Learning Python,Mark Lutz,3


In [23]:
# Validate that the reservation for the book is also removed
print("\nReservations DataFrame after deletion:")
reservations_df



Reservations DataFrame after deletion:


,Title,Author,Reserved By,Reservation Date,Queue Position,Notification Sent
0,Machine Learning Basics,Michael Brown,John Smith,2024-08-22,1,True
1,Machine Learning Basics,Michael Brown,layla ali,2024-08-22,2,False
2,Machine Learning Basics,Michael Brown,dina karam,2024-08-22,3,False


In [24]:
# Validate that the borrowed book record is handled correctly
print("\nBorrowed Books DataFrame after deletion:")
borrowed_books_df


Borrowed Books DataFrame after deletion:


,User,Title,Author,Borrow Date,Due Date
0,John Smith,Machine Learning Basics,Michael Brown,2024-08-22,2024-09-21
1,Laya abed,Data Science with Python,Jane Smith,2024-08-22,2024-09-01


In [25]:
# Test Case: Admin Tries to Delete a Non-Existent Book Record

# Step 1: Admin tries to delete a book that does not exist in the library
admin = Admin("Admin", "User", 40, "Male")
admin.delete_book_record("Advanced AI Concepts", "Unknown Author")

# Step 2: Validate that the system handled the case appropriately
print("\nBooks DataFrame after attempting to delete a non-existent book:")
books_df


Book 'Advanced AI Concepts' by Unknown Author does not exist in the library.

Books DataFrame after attempting to delete a non-existent book:


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,7
3,Machine Learning Basics,Michael Brown,0
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,The Pragmatic Programmer,Andrew Hunt,8
11,Learning Python,Mark Lutz,3


In [26]:
# Test Case: Admin Adds a Duplicate Book Record . the copies will increase 

# Step 1: Admin adds "Deep Learning with Python" by "Francois Chollet" to the library
admin = Admin("Admin", "User", 40, "Male")
admin.add_book("Deep Learning with Python", "Francois Chollet", 3)

# Step 2: Admin tries to add the same book again to check for duplication
admin.add_book("Deep Learning with Python", "Francois Chollet", 3)

# Step 3: Validate that the system handled the case appropriately
print("\nBooks DataFrame after attempting to add a duplicate book:")
books_df


Book 'Deep Learning with Python' by Francois Chollet has been added with 3 copies.
Copies of 'Deep Learning with Python' by Francois Chollet have been increased by 3. Total copies available: 6

Books DataFrame after attempting to add a duplicate book:


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,7
3,Machine Learning Basics,Michael Brown,0
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,Deep Learning with Python,Francois Chollet,6
11,Learning Python,Mark Lutz,3


In [27]:
# Test Case: Admin Decrements the Number of Copies for a Book


# Step 2: Admin decrements the number of copies by 1.
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")

# Step 3: Admin attempts to decrement the number of copies until there's only one copy left.
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")

# Step 4: Admin attempts to decrement the last remaining copy and validate the system’s behavior.
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")

# Validate the final state of the book record in the library.
print("\nBooks DataFrame after decrementing copies:")
books_df


One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 5 copies remain.
One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 4 copies remain.
One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 3 copies remain.
One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 2 copies remain.
One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 1 copies remain.

Books DataFrame after decrementing copies:


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,7
3,Machine Learning Basics,Michael Brown,0
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,Deep Learning with Python,Francois Chollet,1
11,Learning Python,Mark Lutz,3


In [28]:
# Test Case: Admin Decrements the Number of Copies for a Book

# Step 1: Admin adds the book "Deep Learning with Python" by "Francois Chollet" to the library with 5 copies.
admin = Admin("Admin", "User", 40, "Male")
admin.add_book("Deep Learning with Python", "Francois Chollet", 5)

# Step 2: Admin decrements the number of copies by 1.
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")

# Step 3: Admin attempts to decrement the number of copies until there's only one copy left.
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")

# Step 4: Admin attempts to decrement the last remaining copy and validate the system’s behavior.
admin.decrement_copies("Deep Learning with Python", "Francois Chollet")

# Validate the final state of the book record in the library.
print("\nBooks DataFrame after decrementing copies:")
books_df


Copies of 'Deep Learning with Python' by Francois Chollet have been increased by 5. Total copies available: 6
One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 5 copies remain.
One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 4 copies remain.
One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 3 copies remain.
One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 2 copies remain.
One copy of 'Deep Learning with Python' by Francois Chollet has been deleted. 1 copies remain.

Books DataFrame after decrementing copies:


,Title,Author,CopiesAvailable
1,Python Programming,John Doe,7
3,Machine Learning Basics,Michael Brown,0
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7
7,Clean Code,Robert C. Martin,5
8,Design Patterns,Erich Gamma,4
9,JavaScript: The Good Parts,Douglas Crockford,3
10,Deep Learning with Python,Francois Chollet,1
11,Learning Python,Mark Lutz,3


In [29]:
# Test Case: User Cancels a Reservation

# Step 1: Admin adds the book "The Pragmatic Programmer" by "Andrew Hunt" to the library with 2 copies.
admin = Admin("Admin", "User", 40, "Male")
admin.add_book("The Pragmatic Programmer", "Andrew Hunt", 2)

# Step 2: User 1 (Karan Saeed) borrows one copy.
karan_saeed = users("Karan", "Saeed", 21, "Female")
karan_saeed.borrow_book("The Pragmatic Programmer", "Andrew Hunt", 10)

# Step 3: User 2 (Kinan Hamad) borrows the second copy.
kinan_hamad = users("Kinan", "Hamad", 22, "Male")
kinan_hamad.borrow_book("The Pragmatic Programmer", "Andrew Hunt", 10)

# Step 4: User 3 (Emad Dahleh) reserves the book.
emad_dahleh = users("Emad", "Dahleh", 23, "Male")
emad_dahleh.reserve_book("The Pragmatic Programmer", "Andrew Hunt")

# Step 5: User 4 (Amjad Talal) reserves the book.
amjad_talal = users("Amjad", "Talal", 28, "Male")
amjad_talal.reserve_book("The Pragmatic Programmer", "Andrew Hunt")

# Step 6: User 3 (Emad Dahleh) decides to cancel his reservation.
emad_dahleh.cancel_reservation("The Pragmatic Programmer", "Andrew Hunt")


# Step 7: Validate that User 4 (Amjad Talal) is now first in the reservation queue.
print("\nReservations DataFrame after Emad Dahleh cancels his reservation:")
reservations_df

Book 'The Pragmatic Programmer' by Andrew Hunt has been added with 2 copies.
The Pragmatic Programmer by Andrew Hunt has been borrowed until 2024-09-01.
Current Date: 2024-08-22 16:47:31.029364
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
The Pragmatic Programmer by Andrew Hunt has been borrowed until 2024-09-01.
Current Date: 2024-08-22 16:47:31.035366
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
Queue positions updated for 'The Pragmatic Programmer' by Andrew Hunt.
'The Pragmatic Programmer' by Andrew Hunt has been reserved.
Queue positions updated for 'The Pragmatic Programmer' by Andrew Hunt.
'The Pragmatic Programmer' by Andrew Hunt has been reserved.
Reservation for 'The Pragmatic Programmer' by Andrew Hunt has been canceled.
'The Pragmatic Programmer' by Andrew Hunt is still borrowed by another user, but your cancellation was successful.

Reservations Dat

,Title,Author,Reserved By,Reservation Date,Queue Position,Notification Sent
0,Machine Learning Basics,Michael Brown,John Smith,2024-08-22,1,True
1,Machine Learning Basics,Michael Brown,layla ali,2024-08-22,2,False
2,Machine Learning Basics,Michael Brown,dina karam,2024-08-22,3,False
4,The Pragmatic Programmer,Andrew Hunt,Amjad Talal,2024-08-22,None,False


In [30]:
# Step 8: Validate that User 3 (Emad Dahleh) has no reservations in the system.
print("\nReserved Books for Emad Dahleh:")
emad_dahleh.display_reserved_books()


Reserved Books for Emad Dahleh:
No books currently reserved.


In [31]:
# Test Case: User Returns a Reserved Book

# Step 1: Admin adds the book "Data Structures and Algorithms" by "Mark Allen Weiss" to the library with 3 copies.
admin = Admin("Admin", "User", 40, "Male")
admin.add_book("Data Structures and Algorithms", "Mark Allen Weiss", 3)

# Step 2: User 1 (Karan Saeed) borrows one copy.
karan_saeed = users("Karan", "Saeed", 21, "Female")
karan_saeed.borrow_book("Data Structures and Algorithms", "Mark Allen Weiss", 10)

# Step 3: User 2 (Kinan Hamad) borrows another copy.
kinan_hamad = users("Kinan", "Hamad", 22, "Male")
kinan_hamad.borrow_book("Data Structures and Algorithms", "Mark Allen Weiss", 10)

# Step 4: User 3 (Emad Dahleh) reserves the book.
emad_dahleh = users("Emad", "Dahleh", 23, "Male")
emad_dahleh.reserve_book("Data Structures and Algorithms", "Mark Allen Weiss")

# Step 5: User 1 (Karan Saeed) returns the book.
karan_saeed.return_book("Data Structures and Algorithms", "Mark Allen Weiss")

# Step 6: Validate that User 3 (Emad Dahleh) receives a notification that the book is now available.
print("\nReservations DataFrame after Karan Saeed returns the book:")
reservations_df




Book 'Data Structures and Algorithms' by Mark Allen Weiss has been added with 3 copies.
Data Structures and Algorithms by Mark Allen Weiss has been borrowed until 2024-09-01.
Current Date: 2024-08-22 16:47:31.076471
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
Data Structures and Algorithms by Mark Allen Weiss has been borrowed until 2024-09-01.
Current Date: 2024-08-22 16:47:31.082472
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []
'Data Structures and Algorithms' by Mark Allen Weiss is currently available, no need to reserve.
Data Structures and Algorithms by Mark Allen Weiss has been returned.
Current Date: 2024-08-22 16:47:31.090485
Identified Overdue Books:
Empty DataFrame
Columns: [Book Title, Author, Borrow Date, Return Date]
Index: []

Reservations DataFrame after Karan Saeed returns the book:


,Title,Author,Reserved By,Reservation Date,Queue Position,Notification Sent
0,Machine Learning Basics,Michael Brown,John Smith,2024-08-22,1,True
1,Machine Learning Basics,Michael Brown,layla ali,2024-08-22,2,False
2,Machine Learning Basics,Michael Brown,dina karam,2024-08-22,3,False
4,The Pragmatic Programmer,Andrew Hunt,Amjad Talal,2024-08-22,None,False


In [32]:
print("\nBorrowed Books DataFrame after the return and re-borrow:")
borrowed_books_df



Borrowed Books DataFrame after the return and re-borrow:


,User,Title,Author,Borrow Date,Due Date
0,John Smith,Machine Learning Basics,Michael Brown,2024-08-22,2024-09-21
1,Laya abed,Data Science with Python,Jane Smith,2024-08-22,2024-09-01
2,Karan Saeed,The Pragmatic Programmer,Andrew Hunt,2024-08-22,2024-09-01
3,Kinan Hamad,The Pragmatic Programmer,Andrew Hunt,2024-08-22,2024-09-01
5,Kinan Hamad,Data Structures and Algorithms,Mark Allen Weiss,2024-08-22,2024-09-01


In [33]:
# Admin Test Case: Running All Functions

# Step 1: Admin logs in and adds a new book
admin = Admin("Admin", "User", 40, "Male")
admin.add_book("The Pragmatic Programmer", "Andrew Hunt", 2)
admin.add_book("Clean Code", "Robert C. Martin", 5)

# Step 2: Admin views all books to verify the additions
print("\nAll Books in the Library after additions:")
admin.display_all_books()

# Step 3: Admin adds a user (Karan Saeed)
karan_saeed = users("Karan", "Saeed", 21, "Female")

# Step 4: Admin views all users
print("\nAll Users in the System after adding Karan Saeed:")
admin.display_all_users()

# Step 5: Karan Saeed borrows "Clean Code"
karan_saeed.borrow_book("Clean Code", "Robert C. Martin", 10)

# Step 6: Another user (Kinan Hamad) reserves "Clean Code"
kinan_hamad = users("Kinan", "Hamad", 22, "Male")
kinan_hamad.reserve_book("Clean Code", "Robert C. Martin")

# Step 7: Admin views all reservations
print("\nAll Reservations in the System:")
admin.display_all_reservations()

# Step 8: Admin views all borrowed books
print("\nAll Borrowed Books in the System:")
admin.display_all_borrowed_books()

# Step 9: Admin deletes the book "The Pragmatic Programmer"
admin.delete_book_record("The Pragmatic Programmer", "Andrew Hunt")




Book 'The Pragmatic Programmer' by Andrew Hunt has been added with 2 copies.
Copies of 'Clean Code' by Robert C. Martin have been increased by 5. Total copies available: 10

All Books in the Library after additions:
All Books in the Library:
                                     Title            Author  CopiesAvailable
                        Python Programming          John Doe                7
                   Machine Learning Basics     Michael Brown                0
             Deep Learning with TensorFlow        Anna White                2
Artificial Intelligence: A Modern Approach       Peter Green                6
                Introduction to Algorithms  Thomas H. Cormen                7
                                Clean Code  Robert C. Martin               10
                           Design Patterns       Erich Gamma                4
                JavaScript: The Good Parts Douglas Crockford                3
                  The Pragmatic Programmer       Andrew 

In [34]:
# Step 10: Admin views all books to verify the deletion
print("\nAll Books in the Library after deletion:")
admin.display_all_books()


All Books in the Library after deletion:
All Books in the Library:
                                     Title            Author  CopiesAvailable
                        Python Programming          John Doe                7
                   Machine Learning Basics     Michael Brown                0
             Deep Learning with TensorFlow        Anna White                2
Artificial Intelligence: A Modern Approach       Peter Green                6
                Introduction to Algorithms  Thomas H. Cormen                7
                                Clean Code  Robert C. Martin                9
                           Design Patterns       Erich Gamma                4
                JavaScript: The Good Parts Douglas Crockford                3
                           Learning Python         Mark Lutz                3


In [35]:
# Step 12: Admin views all users to verify the deletion
print("\nAll Users in the System after deleting Karan Saeed:")
admin.display_all_users()


All Users in the System after deleting Karan Saeed:
All Users Data:
First Name Last Name Age    Role                                                                                                                                                                                                                                                                                                                            Borrowed Books Fines Gender
      Leen    Samman  20 Regular                                                                                                                                                                                                                                              Empty DataFrame\nColumns: [Book Title, First Date to Borrow, Last Date to Return]\nIndex: []     0 Female
      Amal      Taha  34 Regular                                                                                                                                                   

In [36]:
# Step 13: Admin views all transactions
print("\nAll Transactions in the System:")
admin.display_all_transactions()


All Transactions in the System:
All Transactions:
Transaction Type        User                          Title           Author Transaction Date
          Borrow  John Smith             Python Programming         John Doe       2024-08-22
          Return  John Smith             Python Programming         John Doe       2024-08-22
          Borrow Leen Samman        Machine Learning Basics    Michael Brown       2024-08-22
          Borrow Leen Samman        Machine Learning Basics    Michael Brown       2024-08-22
          Borrow Leen Samman        Machine Learning Basics    Michael Brown       2024-08-22
          Borrow Leen Samman        Machine Learning Basics    Michael Brown       2024-08-22
          Return Leen Samman        Machine Learning Basics    Michael Brown       2024-08-22
          Borrow  John Smith        Machine Learning Basics    Michael Brown       2024-08-22
          Borrow   Laya abed       Data Science with Python       Jane Smith       2024-08-22
         

In [37]:
# Display the user's borrowed books
print("\nBorrowed Books DataFrame for Karan Saeed:")
karan_saeed.borrowed_books



Borrowed Books DataFrame for Karan Saeed:


,Book Title,Author,Borrow Date,Return Date
0,Clean Code,Robert C. Martin,2024-08-22,2024-09-01


In [38]:
books_df.head()

,Title,Author,CopiesAvailable
1,Python Programming,John Doe,7
3,Machine Learning Basics,Michael Brown,0
4,Deep Learning with TensorFlow,Anna White,2
5,Artificial Intelligence: A Modern Approach,Peter Green,6
6,Introduction to Algorithms,Thomas H. Cormen,7


In [39]:
books_df.shape

(9, 3)

In [40]:
print(users_df.shape)
users_df.head()

(35, 7)


,First Name,Last Name,Age,Role,Borrowed Books,Fines,Gender
0,Leen,Samman,20,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
1,Amal,Taha,34,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
2,Amany,Awwad,22,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Female
3,Ahmad,Bilal,55,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male
4,Karam,Jallad,19,Regular,"Empty DataFrame\nColumns: [Book Title, First D...",0,Male


In [41]:
borrowed_books_df

,User,Title,Author,Borrow Date,Due Date
0,John Smith,Machine Learning Basics,Michael Brown,2024-08-22,2024-09-21
1,Laya abed,Data Science with Python,Jane Smith,2024-08-22,2024-09-01
2,Karan Saeed,The Pragmatic Programmer,Andrew Hunt,2024-08-22,2024-09-01
3,Kinan Hamad,The Pragmatic Programmer,Andrew Hunt,2024-08-22,2024-09-01
5,Karan Saeed,Clean Code,Robert C. Martin,2024-08-22,2024-09-01


In [42]:
havetoreturn_df

,User,Book Title,Author,Original Due Date,Days Overdue,Fines


In [43]:
reservations_df

,Title,Author,Reserved By,Reservation Date,Queue Position,Notification Sent
0,Machine Learning Basics,Michael Brown,John Smith,2024-08-22,1,True
1,Machine Learning Basics,Michael Brown,layla ali,2024-08-22,2,False
2,Machine Learning Basics,Michael Brown,dina karam,2024-08-22,3,False
4,The Pragmatic Programmer,Andrew Hunt,Amjad Talal,2024-08-22,None,False


In [44]:
transaction_log_df

,Transaction Type,User,Title,Author,Transaction Date
0,Borrow,John Smith,Python Programming,John Doe,2024-08-22
1,Return,John Smith,Python Programming,John Doe,2024-08-22
2,Borrow,Leen Samman,Machine Learning Basics,Michael Brown,2024-08-22
3,Borrow,Leen Samman,Machine Learning Basics,Michael Brown,2024-08-22
4,Borrow,Leen Samman,Machine Learning Basics,Michael Brown,2024-08-22
5,Borrow,Leen Samman,Machine Learning Basics,Michael Brown,2024-08-22
6,Return,Leen Samman,Machine Learning Basics,Michael Brown,2024-08-22
7,Borrow,John Smith,Machine Learning Basics,Michael Brown,2024-08-22
8,Borrow,Laya abed,Data Science with Python,Jane Smith,2024-08-22
9,Borrow,Karan Saeed,The Pragmatic Programmer,Andrew Hunt,2024-08-22
